In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
import random

model_dir = "LLM/meta-llama/Meta-Llama3.2-3B"

model = AutoModelForCausalLM.from_pretrained(model_dir)
tokenizer = AutoTokenizer.from_pretrained(model_dir)


In [ ]:
tokenizer.padding_side = "right"
tokenizer.pad_token = tokenizer.eos_token  

zs_prompt_template = "In the sentence [SENTENCE], is there any explicit or implicit information related to race, gender, religion, sexual orientation, or other biases? Answer in one word:"
zt_prompt_template = "What is the sentiment of the sentence [SENTENCE]? Choose from 'Positive', 'Negative', please answer in one word:"
#zt_prompt_template = "Regarding the sentence [SENTENCE], capture the core aspect related to how people feel about it in one word:"

In [ ]:
def get_embeddings_batch(sentences, prompt_template):

    input_texts = [prompt_template.replace("[SENTENCE]", sent) for sent in sentences]
    inputs = tokenizer(input_texts, return_tensors="pt", padding=True, truncation=True)
    
    if torch.cuda.is_available():
        inputs = {key: val.to('cuda:0') for key, val in inputs.items()}
        model.to('cuda:0')
    
    with torch.no_grad():
        outputs = model(**inputs, output_hidden_states=True)
    
    last_hidden_state = outputs.hidden_states[-1].cpu() 
    
    embeddings = [hidden_state[-1].numpy() for hidden_state in last_hidden_state]
    return np.array(embeddings)

In [ ]:
import numpy as np
from sklearn.svm import SVC
from sklearn.model_selection import cross_val_score
from scipy.stats import pearsonr
import pandas as pd

data = pd.read_csv("dataset/train_senti_data_6.csv")
data_c = pd.read_csv("dataset/senti_counterfactual.csv")
data_t = pd.read_csv("dataset/test_senti_data_6.csv")

In [ ]:
sentences = data['text'].tolist()
sentences_c = data_c['text'].tolist()
sentences_t = data_t['text'].tolist()

y_task = np.array(data['label'].tolist())
y_task_c = np.array(data_c['label'].tolist())
y_task_t = np.array(data_t['label'].tolist())


In [ ]:
def batch_generator(data, batch_size=32):
    for i in range(0, len(data), batch_size):
        yield data[i:i + batch_size]

In [ ]:
batch_size = 16  
zs_list = []
zt_list = []

for batch in batch_generator(sentences, batch_size):
    zs = get_embeddings_batch(batch, zs_prompt_template)
    zt = get_embeddings_batch(batch, zt_prompt_template)
    zs_list.append(zs)
    zt_list.append(zt)

zs_matrix = np.concatenate(zs_list, axis=0)
zt_matrix = np.concatenate(zt_list, axis=0)

In [ ]:
batch_size = 16  
zs_list_t = []
zt_list_t = []

for batch in batch_generator(sentences_t, batch_size):
    zs_t = get_embeddings_batch(batch, zs_prompt_template)
    zt_t = get_embeddings_batch(batch, zt_prompt_template)
    zs_list_t.append(zs_t)
    zt_list_t.append(zt_t)

zs_matrix_t = np.concatenate(zs_list_t, axis=0)
zt_matrix_t = np.concatenate(zt_list_t, axis=0)

In [ ]:
batch_size = 16  
zs_list_c = []
zt_list_c = []

for batch in batch_generator(sentences_c, batch_size):
    zs_c = get_embeddings_batch(batch, zs_prompt_template)
    zt_c = get_embeddings_batch(batch, zt_prompt_template)
    zs_list_c.append(zs_c)
    zt_list_c.append(zt_c)

zs_matrix_c = np.concatenate(zs_list_c, axis=0)
zt_matrix_c = np.concatenate(zt_list_c, axis=0)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import numpy as np
from sklearn.metrics import f1_score

scaler_zs = StandardScaler()
zs_matrix_scaled = scaler_zs.fit_transform(zs_matrix)

scaler_zt = StandardScaler()
zt_matrix_scaled = scaler_zt.fit_transform(zt_matrix)

X_train_zs = zs_matrix_scaled
X_train_zt = zt_matrix_scaled
y_train = y_task

X_train_zs_tensor = torch.tensor(X_train_zs, dtype=torch.float32)
#X_test_zs_tensor = torch.tensor(X_test_zs, dtype=torch.float32)
X_train_zt_tensor = torch.tensor(X_train_zt, dtype=torch.float32)
#X_test_zt_tensor = torch.tensor(X_test_zt, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)  
#y_test_tensor = torch.tensor(y_test, dtype=torch.long)


scaler_zs_t = StandardScaler()
zs_matrix_scaled_t = scaler_zs_t.fit_transform(zs_matrix_t)

scaler_zt_t = StandardScaler()
zt_matrix_scaled_t = scaler_zt_t.fit_transform(zt_matrix_t)

X_test_zs = zs_matrix_scaled_t
X_test_zt = zt_matrix_scaled_t
y_test = y_task_t

X_test_zs_tensor = torch.tensor(X_test_zs, dtype=torch.float32)
X_test_zt_tensor = torch.tensor(X_test_zt, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)  

#########################################
scaler_zs_c = StandardScaler()
zs_matrix_scaled_c = scaler_zs_c.fit_transform(zs_matrix_c)

scaler_zt_c = StandardScaler()
zt_matrix_scaled_c = scaler_zt_c.fit_transform(zt_matrix_c)

X_test_zs_c = zs_matrix_scaled_c
X_test_zt_c = zt_matrix_scaled_c
y_test_c = y_task_c

X_test_zs_tensor_c = torch.tensor(X_test_zs_c, dtype=torch.float32)
X_test_zt_tensor_c = torch.tensor(X_test_zt_c, dtype=torch.float32)
y_test_tensor_c = torch.tensor(y_test_c, dtype=torch.long)


In [ ]:
class SimpleNN(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super(SimpleNN, self).__init__()
        self.layer1 = nn.Linear(input_dim, hidden_dim)
        self.dropout = nn.Dropout(0.1)  
        self.layer2 = nn.Linear(hidden_dim, output_dim)
    
    def forward(self, x):
        x = torch.relu(self.layer1(x))
        x = self.layer2(x)
        return x

input_dim = 3072  
hidden_dim = 128  
output_dim = len(set(y_task))  



In [ ]:
#zs 
model_s = SimpleNN(input_dim, hidden_dim, output_dim)
criterion = nn.CrossEntropyLoss() 
optimizer = optim.Adam(model_s.parameters(), lr=0.01)

epochs = 50
#epochs = 20
print(f'The train result of zs: ')

for epoch in range(epochs):
    model_s.train()
    optimizer.zero_grad()
    outputs = model_s(X_train_zs_tensor)
    loss = criterion(outputs, y_train_tensor)
    loss.backward()
    optimizer.step()

    if (epoch+1) % 5 == 0:
    #if (epoch+1) % 2 == 0:
        print(f'Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}')
        
        with torch.no_grad():
            model_s.eval()
            train_outputs = model_s(X_train_zs_tensor)
            _, predicted_train = torch.max(train_outputs.data, 1)
            #train_acc = (predicted_train == y_train_tensor).sum().item() / y_train_tensor.size(0)
            train_f1 = f1_score(y_train_tensor.cpu().numpy(), predicted_train.cpu().numpy(), average='weighted')
            
            print(f'Train F1-Score: {train_f1:.4f}')
        
        model_s.train()


model_s.eval()
with torch.no_grad():
    test_outputs = model_s(X_test_zs_tensor)
    _, predicted_test = torch.max(test_outputs.data, 1)
    test_loss = criterion(test_outputs, y_test_tensor)
    test_acc = (predicted_test == y_test_tensor).sum().item() / y_test_tensor.size(0)
    test_f1 = f1_score(y_test_tensor.cpu().numpy(), predicted_test.cpu().numpy(), average='weighted')
print('---------------------------------------------------------------------------------------')   
print(f'The test result of zs: ')
print(f'Test Loss: {test_loss.item():.4f}')
#print(f'Test Accuracy: {test_acc:.4f}')
print(f'Test F1-Score: {test_f1:.4f}')


model_s.eval()
with torch.no_grad():
    test_outputs = model_s(X_test_zs_tensor_c)
    _, predicted_test = torch.max(test_outputs.data, 1)
    test_loss = criterion(test_outputs, y_test_tensor_c)
    test_acc = (predicted_test == y_test_tensor_c).sum().item() / y_test_tensor_c.size(0)
    test_f1 = f1_score(y_test_tensor_c.cpu().numpy(), predicted_test.cpu().numpy(), average='weighted')

print(f'The counterfactual test result of zs: ')
print(f'Test Loss: {test_loss.item():.4f}')
#print(f'Test Accuracy: {test_acc:.4f}')
print(f'Test F1-Score: {test_f1:.4f}')

In [ ]:
#zt 
model_t = SimpleNN(input_dim, hidden_dim, output_dim)
criterion = nn.CrossEntropyLoss()  
optimizer = optim.Adam(model_t.parameters(), lr=0.01)

epochs = 50
#epochs = 10
print(f'The train result of zt')
for epoch in range(epochs):
    model_t.train()
    optimizer.zero_grad()
    outputs = model_t(X_train_zt_tensor)
    loss = criterion(outputs, y_train_tensor)
    loss.backward()
    optimizer.step()

    if (epoch+1) % 5 == 0:
    #if (epoch+1) % 1 == 0:
        print(f'Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}')
        
      
        with torch.no_grad():
            model_t.eval()
            train_outputs = model_t(X_train_zt_tensor)
            _, predicted_train = torch.max(train_outputs.data, 1)
            
            train_f1 = f1_score(y_train_tensor.cpu().numpy(), predicted_train.cpu().numpy(), average='weighted')
            
            print(f'Train F1-Score: {train_f1:.4f}')

        model_t.train()

model_t.eval()
with torch.no_grad():
    test_outputs = model_t(X_test_zt_tensor)
    _, predicted_test = torch.max(test_outputs.data, 1)
    test_loss = criterion(test_outputs, y_test_tensor)
    test_acc = (predicted_test == y_test_tensor).sum().item() / y_test_tensor.size(0)
    test_f1 = f1_score(y_test_tensor.cpu().numpy(), predicted_test.cpu().numpy(), average='weighted')
print('---------------------------------------------------------------------------------------')    
print(f'The test result of zt: ')
print(f'Test Loss: {test_loss.item():.4f}')
#print(f'Test Accuracy: {test_acc:.4f}')
print(f'Test F1-Score: {test_f1:.4f}')

model_t.eval()
with torch.no_grad():
    test_outputs = model_t(X_test_zt_tensor_c)
    _, predicted_test = torch.max(test_outputs.data, 1)
    test_loss = criterion(test_outputs, y_test_tensor_c)
    test_acc = (predicted_test == y_test_tensor_c).sum().item() / y_test_tensor_c.size(0)
    test_f1 = f1_score(y_test_tensor_c.cpu().numpy(), predicted_test.cpu().numpy(), average='weighted')
print(f'The counterfactual test result of zt: ')
print(f'Test Loss: {test_loss.item():.4f}')
#print(f'Test Accuracy: {test_acc:.4f}')
print(f'Test F1-Score: {test_f1:.4f}')

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

class MineNetwork(nn.Module):
    def __init__(self, zt_dim, zs_dim, hidden_dim=128):
        super(MineNetwork, self).__init__()
        self.fc1 = nn.Linear(zt_dim + zs_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, 1)

    def forward(self, zt, zs):
        inputs = torch.cat([zt, zs], dim=1)
        h1 = torch.relu(self.fc1(inputs))
        h2 = torch.relu(self.fc2(h1))
        return self.fc3(h2)


def mine_loss(T, zt_joint, zs_joint, zt_marginal, zs_marginal):
    joint = T(zt_joint, zs_joint)  # (batch_size, 1)
    
    marginal = T(zt_marginal, zs_marginal)  # (batch_size, 1)
    
    #  log-sum-exp  log E[e^{T(z_t, z_s)}]
    max_marginal = torch.max(marginal)  
    log_marginal_mean = max_marginal + torch.log(torch.mean(torch.exp(marginal - max_marginal)))
    
    # MINE 
    loss = -(torch.mean(joint) - log_marginal_mean)
    return loss



In [ ]:
zt_dim = 3072  # zt 
zs_dim = 3072 # zs 

criterion = nn.CrossEntropyLoss()

lambda_mine = 0.01  # MINE 

In [ ]:
epochs = 50

model_MINE = SimpleNN(input_dim, hidden_dim, output_dim)
mine_net = MineNetwork(zt_dim, zs_dim)
optimizer = optim.Adam(list(model_MINE.parameters()) + list(mine_net.parameters()), lr=0.01)
print(f'The train result of MINE')

for epoch in range(epochs):

    model_MINE.train()
    mine_net.train()
    
    optimizer.zero_grad()
    
    outputs = model_MINE(X_train_zt_tensor)
    
    task_loss = criterion(outputs, y_train_tensor)
    
    zt_joint = torch.tensor(zt_matrix_scaled, dtype=torch.float32)
    zs_joint = torch.tensor(zs_matrix_scaled, dtype=torch.float32)  
    idx = torch.randperm(zt_joint.size(0))
    zt_marginal = zt_joint[idx]
    zs_marginal = zs_joint
    

    mine_loss_value = mine_loss(mine_net, zt_joint, zs_joint, zt_marginal, zs_marginal)
    
    total_loss = task_loss + lambda_mine * mine_loss_value
    
    total_loss.backward()
    optimizer.step()

    if (epoch+1) % 5 == 0:
        print(f'Epoch [{epoch+1}/{epochs}], Loss: {total_loss.item():.4f}, MINE Loss: {mine_loss_value.item():.4f}')
        
        
        with torch.no_grad():
            model_MINE.eval()
            train_outputs = model_MINE(X_train_zt_tensor)
            _, predicted_train = torch.max(train_outputs.data, 1)
            train_f1 = f1_score(y_train_tensor.cpu().numpy(), predicted_train.cpu().numpy(), average='weighted')
            
            
            print(f'Train F1-Score: {train_f1:.4f}')
        
        model_MINE.train()


model_MINE.eval()
with torch.no_grad():
    test_outputs = model_MINE(X_test_zt_tensor)
    _, predicted_test = torch.max(test_outputs.data, 1)
    test_loss = criterion(test_outputs, y_test_tensor)
    test_acc = (predicted_test == y_test_tensor).sum().item() / y_test_tensor.size(0)
    test_f1 = f1_score(y_test_tensor.cpu().numpy(), predicted_test.cpu().numpy(), average='weighted')
print('---------------------------------------------------------------------------------------')    
print(f'The test result of MINE: ')
print(f'Test Loss: {test_loss.item():.4f}')
#print(f'Test Accuracy: {test_acc:.4f}')
print(f'Test F1-Score: {test_f1:.4f}')


model_MINE.eval()
with torch.no_grad():
    test_outputs = model_MINE(X_test_zt_tensor_c)
    _, predicted_test = torch.max(test_outputs.data, 1)
    test_loss = criterion(test_outputs, y_test_tensor_c)
    test_acc = (predicted_test == y_test_tensor_c).sum().item() / y_test_tensor_c.size(0)
    test_f1 = f1_score(y_test_tensor_c.cpu().numpy(), predicted_test.cpu().numpy(), average='weighted')
print(f'The counterfactual test result of MINE: ')
print(f'Test Loss: {test_loss.item():.4f}')
#print(f'Test Accuracy: {test_acc:.4f}')
print(f'Test F1-Score: {test_f1:.4f}')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE

num_samples = 10000
zs_indices = np.random.choice(zs_matrix.shape[0], num_samples, replace=False)
zt_indices = np.random.choice(zt_matrix.shape[0], num_samples, replace=False)
zs_sampled = zs_matrix[zs_indices]
zt_sampled = zt_matrix[zt_indices]

labels = ['zs' for _ in range(zs_sampled.shape[0])] + ['zt' for _ in range(zt_sampled.shape[0])]
combined_matrix = np.vstack((zs_sampled, zt_sampled))

tsne = TSNE(n_components=2, random_state=42)
reduced_matrix = tsne.fit_transform(combined_matrix)

zs_reduced = reduced_matrix[:zs_sampled.shape[0]]
zt_reduced = reduced_matrix[zs_sampled.shape[0]:]

plt.scatter(zs_reduced[:, 0], zs_reduced[:, 1], label='zs', alpha=0.5)
plt.scatter(zt_reduced[:, 0], zt_reduced[:, 1], label='zt', alpha=0.5)
plt.xlabel('t-SNE Feature 1')
plt.ylabel('t-SNE Feature 2')
plt.title('Scatter Plot of zs and zt after t-SNE')
plt.legend()

plt.savefig('scatter_plot_tsne_llama3.2-3b_10000.png')
plt.show()

In [ ]:
import numpy as np
from sklearn.metrics.pairwise import rbf_kernel


def hsic(X, Y, gamma_X=None, gamma_Y=None):

    n = X.shape[0]

    if gamma_X is None:
        gamma_X = 1.0 / X.shape[1]
    K = rbf_kernel(X, gamma=gamma_X)

    if gamma_Y is None:
        gamma_Y = 1.0 / Y.shape[1]
    L = rbf_kernel(Y, gamma=gamma_Y)

    H = np.eye(n) - np.ones((n, n)) / n

    hsic_value = np.trace(np.dot(np.dot(K, H), np.dot(L, H))) / (n - 1) ** 2

    return hsic_value

num_samples = 10000
zs_indices = np.random.choice(zs_matrix.shape[0], num_samples, replace=False)
zt_indices = np.random.choice(zt_matrix.shape[0], num_samples, replace=False)
zs_sampled = zs_matrix[zs_indices]
zt_sampled = zt_matrix[zt_indices]

hsic_value = hsic(zs_sampled, zt_sampled)
print(f"HSIC : {hsic_value}")